# 13. Normalize Numeric Features & Train/Test Split


## 13.1 Import Libraries


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA_DIR = r"A:\pc\Desktop\Code\data science\sadop\Data"

feature_dataset_path = rf"{DATA_DIR}\ml_features.csv"
df = pd.read_csv(feature_dataset_path)

# Fix estimated_rows NaN → 0
df["estimated_rows"] = df["estimated_rows"].fillna(0)

df.head()

,query,query_time,rows_returned,has_sum,has_group_by,has_where,tables_count,query_length,cpu_usage,estimated_rows,uses_index,full_table_scan,uses_filesort,uses_temp_table,is_slow
0,SELECT *\nFROM user\nWHERE user_id IN (\n S...,0.850983,16520,0,0,1,1,189,25.0,54923,1,1,0,0,1
1,"SELECT u.user_id,\n (\n SELECT S...",0.745494,20000,1,0,1,2,217,0.0,20207,1,0,0,0,1
2,SELECT *\nFROM user u\nWHERE EXISTS (\n SEL...,0.574111,16121,0,0,1,2,184,0.0,54923,1,1,0,0,0
3,"SELECT u.user_id, t.transaction_date, t.amount...",1.684971,250000,0,0,0,3,185,0.0,34725,1,0,1,1,1
4,"SELECT u.user_id,\n (\n SELECT S...",0.599545,20000,1,0,1,2,217,0.0,20207,1,0,0,0,0


## 3.2 Log-Transform query_time


In [2]:
# Avoid log(0) by adding a tiny value
df['query_time_log'] = np.log1p(df['query_time'])
df[['query_time', 'query_time_log']].describe()


,query_time,query_time_log
count,19975.000000,19975.000000
mean,0.878726,0.464602
std,1.517166,0.496284
min,0.000000,0.000000
25%,0.086555,0.083012
50%,0.495865,0.402705
75%,0.719669,0.542132
max,9.993443,2.397299


## 3.3 Select Features & Targets


In [3]:
FEATURES = [
    # original
    "rows_returned",
    "tables_count",
    "query_length",
    "has_sum",
    "has_group_by",
    "has_where",
    "cpu_usage",
    # EXPLAIN
    "estimated_rows",
    "uses_index",
    "full_table_scan",
    "uses_filesort",
    "uses_temp_table",
]

X   = df[FEATURES]
y   = df["is_slow"]

## 3.4 Train/Test Split


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", X_train.shape)
print("Test  size:", X_test.shape)
print(f"\nClass balance (train):\n{y_train.value_counts(normalize=True).round(3)}")


Train size: (15980, 12)
Test  size: (3995, 12)

Class balance (train):
is_slow
0    0.561
1    0.439
Name: proportion, dtype: float64


## 3.5 Normalize numeric features

In [5]:
scaler       = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)



## 3.6 Save Preprocessed Datasets 


In [6]:
pd.DataFrame(X_train_scaled, columns=FEATURES).to_csv(rf"{DATA_DIR}\X_train.csv", index=False)
pd.DataFrame(X_test_scaled,  columns=FEATURES).to_csv(rf"{DATA_DIR}\X_test.csv",  index=False)
y_train.to_csv(rf"{DATA_DIR}\y_train.csv", index=False)
y_test.to_csv( rf"{DATA_DIR}\y_test.csv",  index=False)

print("✅ Preprocessed datasets saved")

✅ Preprocessed datasets saved
